In [1]:
import pandas as pd
import re
import string

df = pd.read_csv("../data/raw/tickets.csv")
sample_texts = df["instruction"].sample(5, random_state=1).tolist()

print("Sample raw tickets:")
for t in sample_texts:
    print(" -", t)

Sample raw tickets:
 - how can I check bill  #12588?
 - I'm rtying to cancel order {{Order Number}}
 - I do not know what to do to demand my money back
 - assistance to see if there are any news on the refund
 - I don't know how t leave an opinion about a product


In [2]:
for t in sample_texts:
    print("BEFORE:", t)
    print("AFTER :", t.lower())
    print()

BEFORE: how can I check bill  #12588?
AFTER : how can i check bill  #12588?

BEFORE: I'm rtying to cancel order {{Order Number}}
AFTER : i'm rtying to cancel order {{order number}}

BEFORE: I do not know what to do to demand my money back
AFTER : i do not know what to do to demand my money back

BEFORE: assistance to see if there are any news on the refund
AFTER : assistance to see if there are any news on the refund

BEFORE: I don't know how t leave an opinion about a product
AFTER : i don't know how t leave an opinion about a product



In [3]:
# Let's find an example that actually has a placeholder
examples_with_placeholder = df[df["instruction"].str.contains(r"\{\{", regex=True)]["instruction"].head(3)
for t in examples_with_placeholder:
    print("BEFORE:", t)
    cleaned = re.sub(r"\{\{.*?\}\}", " ", t)
    print("AFTER :", cleaned)
    print()

BEFORE: question about cancelling order {{Order Number}}
AFTER : question about cancelling order  

BEFORE: i have a question about cancelling oorder {{Order Number}}
AFTER : i have a question about cancelling oorder  

BEFORE: i need help cancelling puchase {{Order Number}}
AFTER : i need help cancelling puchase  



## Step 3: Remove punctuation

Punctuation like `?`, `!`, `,` rarely helps a TF-IDF model distinguish categories,
so we strip it out. `string.punctuation` is Python's built-in list of all
punctuation characters.


In [4]:
print("Punctuation characters we'll remove:", string.punctuation)

sample = "I can't find my refund, can you help?!"
no_punct = sample.translate(str.maketrans("", "", string.punctuation))
print("\nBEFORE:", sample)
print("AFTER :", no_punct)

Punctuation characters we'll remove: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

BEFORE: I can't find my refund, can you help?!
AFTER : I cant find my refund can you help


In [5]:
sample = "please refund order 88291 as soon as possible"
no_numbers = re.sub(r"\b\d+\b", " ", sample)
print("BEFORE:", sample)
print("AFTER :", no_numbers)

BEFORE: please refund order 88291 as soon as possible
AFTER : please refund order   as soon as possible


In [6]:
sample = "please refund   order    as soon as possible"
tidy = re.sub(r"\s+", " ", sample).strip()
print("BEFORE:", repr(sample))
print("AFTER :", repr(tidy))

BEFORE: 'please refund   order    as soon as possible'
AFTER : 'please refund order as soon as possible'


In [7]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"\{\{.*?\}\}", " ", text)   # remove {{placeholders}}
    text = re.sub(r"\{.*?\}", " ", text)         # remove {single-brace} variants
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # remove URLs
    text = re.sub(r"\S+@\S+", " ", text)          # remove emails
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\b\d+\b", " ", text)          # remove standalone numbers
    text = re.sub(r"\s+", " ", text).strip()       # collapse whitespace

    return text

# Test it on our original samples
for t in sample_texts:
    print("BEFORE:", t)
    print("AFTER :", clean_text(t))
    print()

BEFORE: how can I check bill  #12588?
AFTER : how can i check bill

BEFORE: I'm rtying to cancel order {{Order Number}}
AFTER : im rtying to cancel order

BEFORE: I do not know what to do to demand my money back
AFTER : i do not know what to do to demand my money back

BEFORE: assistance to see if there are any news on the refund
AFTER : assistance to see if there are any news on the refund

BEFORE: I don't know how t leave an opinion about a product
AFTER : i dont know how t leave an opinion about a product



## Step 7: Apply to the whole dataset and compare a few rows

Let's run this on the full dataset and view before/after side-by-side in a table.


In [ ]:
df["clean_instruction"] = df["instruction"].apply(clean_text)

# S
df[["instruction", "clean_instruction"]].sample(5, random_state=2)

,instruction,clean_instruction
12487,can i place a damn order from {{Delivery Count...,can i place a damn order from
3035,I do not know how I can see the cancellation fees,i do not know how i can see the cancellation fees
11704,I want help canceling the standard account,i want help canceling the standard account
20961,help to inform of issues with signu,help to inform of issues with signu
8451,wanna speak to customer support how can i do it,wanna speak to customer support how can i do it
